In [ ]:
import numpy as np
import pandas as pd
import os
import getpass
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

user = getpass.getuser()
root_dir = f"/Users/{user}/Library/Mobile Documents/com~apple~CloudDocs/Desktop/Python_AI_and_Unreal_Workshop/Web Scraping"
csv_path = os.path.join(root_dir, "Global_Gothic_Dataset.csv")
temp_dir = os.path.join(root_dir, "temp_features")
output_csv = os.path.join(root_dir, "vector_comparison_results.csv")

df = pd.read_csv(csv_path)
resnet_feats, clip_feats, valid_paths = [], [], []

print("Loading cached feature files...")

for i, fname in enumerate(df['File_Path']):
    res_cache = os.path.join(temp_dir, f"ResNet_{i}.npy")
    clip_cache = os.path.join(temp_dir, f"CLIP_{i}.npy")

    if os.path.exists(res_cache) and os.path.exists(clip_cache):
        resnet_feats.append(np.load(res_cache))
        clip_feats.append(np.load(clip_cache))
        valid_paths.append(str(fname).strip())

num = len(resnet_feats)
print(f"Loaded {num} feature pairs.")

print("\nRunning t-SNE dimensionality reduction...")

def reduce_dim(data):
    tsne = TSNE(
        n_components=2,
        perplexity=min(15, num - 1),
        init='pca',
        learning_rate='auto',
        random_state=42
    )
    return tsne.fit_transform(np.array(data))

res_2d = reduce_dim(resnet_feats)
clip_2d = reduce_dim(clip_feats)

def norm(v):
    return (v - v.min()) / (v.max() - v.min())

results = pd.DataFrame({
    'file': valid_paths,
    'res_x': norm(res_2d[:, 0]),
    'res_y': norm(res_2d[:, 1]),
    'clip_x': norm(clip_2d[:, 0]),
    'clip_y': norm(clip_2d[:, 1])
})

results.to_csv(output_csv, index=False)
print(f"Saved to: {output_csv}")

plt.clf()
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

sns.scatterplot(data=results, x='res_x', y='res_y', ax=ax1, color='#2c3e50', alpha=0.6)
ax1.set_title('Analysis: ResNet50 (Visual Patterns)', fontsize=14)

sns.scatterplot(data=results, x='clip_x', y='clip_y', ax=ax2, color='#e74c3c', alpha=0.6)
ax2.set_title('Analysis: CLIP (Semantic Meaning)', fontsize=14)

plt.suptitle(f'Comparative Cluster Analysis of Dataset (n={num})', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()
